# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dishaprabhakar2006-blip/Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [8]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

My rule: flag pages that are stale (content_age_days >= 180) AND still visible
(gsc_impressions >= 500). These are pages getting real traffic that nobody has
touched in 6+ months — likely candidates for refresh review.

Reason codes this rule can output:
- stale_visible_page: content_age_days >= 180 and gsc_impressions >= 500

Before encoding it, I checked two signals the rule leans on:
1. Staleness vs decline (linked to the real staleness/refresh flag from the session)
2. CTR vs position tier (linked to the real CTR-fix logic from the session)

In [9]:
# Signal 1: staleness vs decline rate
staleness_check = con.sql(f"""
    SELECT
      CASE
        WHEN date_diff('day', CAST(d.content_created_date AS DATE), CAST('2026-03-31' AS DATE)) < 90 THEN '1_under_90d'
        WHEN date_diff('day', CAST(d.content_created_date AS DATE), CAST('2026-03-31' AS DATE)) < 180 THEN '2_90_180d'
        WHEN date_diff('day', CAST(d.content_created_date AS DATE), CAST('2026-03-31' AS DATE)) < 365 THEN '3_180_365d'
        ELSE '4_over_365d'
      END AS age_bucket,
      COUNT(*) AS n,
      AVG(CASE WHEN gsc_avg_position > 20 THEN 1 ELSE 0 END) AS decline_rate
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_content']} d ON f.content_hash_id = d.content_hash_id
    GROUP BY 1
    ORDER BY 1
""").df()
print(staleness_check)

# Signal 2: CTR vs position tier
ctr_position_check = con.sql(f"""
    SELECT
      CASE
        WHEN gsc_avg_position <= 3 THEN '1_top3'
        WHEN gsc_avg_position <= 10 THEN '2_top10'
        WHEN gsc_avg_position <= 20 THEN '3_top20'
        ELSE '4_beyond20'
      END AS position_bucket,
      COUNT(*) AS n,
      AVG(gsc_clicks / NULLIF(gsc_impressions,0)) AS avg_ctr
    FROM {TABLES['fact_daily']}
    WHERE gsc_impressions > 0
    GROUP BY 1
    ORDER BY 1
""").df()
print(ctr_position_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    age_bucket        n  decline_rate
0  1_under_90d  2116433      0.094562
1    2_90_180d  1133357      0.139974
2   3_180_365d  5588895      0.070686
3  4_over_365d  1002693      0.154107


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_bucket        n   avg_ctr
0          1_top3   727362  0.004756
1         2_top10  1456122  0.003473
2         3_top20   519223  0.002770
3      4_beyond20   908354  0.001289


Verdict 1 (staleness vs decline): MIXED — decline_rate does NOT rise cleanly with
age. It goes 9.5% (under 90d) → 14.0% (90-180d) → 7.1% (180-365d) → 15.4% (over 365d).
If staleness caused decline, we'd expect a steady increase with age — instead it dips
in the middle bucket before rising again. The "older = more declined" assumption is
only partially true here; age alone isn't a clean predictor of decline.

Verdict 2 (CTR vs position): CONFIRMED — avg_ctr drops cleanly and monotonically as
position worsens: 0.48% (top3) → 0.35% (top10) → 0.28% (top20) → 0.13% (beyond20).
This matches the real CTR-fix logic assumption exactly — position really does predict
CTR here.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
baseline = con.sql(f"""
    SELECT
      f.content_hash_id,
      f.client_hash_id,
      f.gsc_impressions,
      f.gsc_clicks,
      f.gsc_avg_position,
      date_diff('day', CAST(d.content_created_date AS DATE), CAST('2026-03-31' AS DATE)) AS content_age_days,
      d.word_count,
      (f.gsc_clicks / NULLIF(f.gsc_impressions,0)) AS ctr
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_content']} d ON f.content_hash_id = d.content_hash_id
    WHERE f.gsc_impressions >= 100
""").df()

baseline["stale"] = (baseline["content_age_days"] >= 180).astype(int)
baseline["visible"] = (baseline["gsc_impressions"] >= 500).astype(int)
baseline["baseline_score"] = baseline["stale"] * baseline["visible"] * baseline["gsc_impressions"]
baseline["reason_code"] = "stale_visible_page"
baseline["action"] = "review_for_refresh"

ranked = baseline.sort_values("baseline_score", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote", len(ranked), "rows")
ranked.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 638608 rows


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,content_age_days,word_count,ctr,stale,visible,baseline_score,reason_code,action
0,content_eadb33b5df496f4a,client_e547b89c05043229,39305,252,2.197507,375,2753,0.006411,1,1,39305,stale_visible_page,review_for_refresh
1,content_34a70fea29d15f24,client_62f4a7e64f5e0096,39003,2,2.764916,278,2639,0.000051,1,1,39003,stale_visible_page,review_for_refresh
2,content_eadb33b5df496f4a,client_e547b89c05043229,38436,271,2.195988,375,2753,0.007051,1,1,38436,stale_visible_page,review_for_refresh
3,content_945d6ff91386c817,client_62f4a7e64f5e0096,37368,0,8.613948,278,2871,0.000000,1,1,37368,stale_visible_page,review_for_refresh
4,content_eadb33b5df496f4a,client_e547b89c05043229,35404,225,2.188397,375,2753,0.006355,1,1,35404,stale_visible_page,review_for_refresh
5,content_eadb33b5df496f4a,client_e547b89c05043229,34817,223,2.181348,375,2753,0.006405,1,1,34817,stale_visible_page,review_for_refresh
6,content_eadb33b5df496f4a,client_e547b89c05043229,34606,235,2.242501,375,2753,0.006791,1,1,34606,stale_visible_page,review_for_refresh
7,content_eadb33b5df496f4a,client_e547b89c05043229,33571,215,2.309046,375,2753,0.006404,1,1,33571,stale_visible_page,review_for_refresh
8,content_fec55986a1868d62,client_73cda7b4e4f265ea,33383,0,0.181500,410,<NA>,0.000000,1,1,33383,stale_visible_page,review_for_refresh
9,content_eadb33b5df496f4a,client_e547b89c05043229,32665,221,2.388857,375,2753,0.006766,1,1,32665,stale_visible_page,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Rows 0,2,4,5,6,10,11,13,16 (content_eadb33b5df496f4a): action=review_for_refresh,
reason=stale_visible_page, confidence=LOW — this is the same content_hash_id
appearing 9 separate times in the top 20 with slightly different impression counts.
This is likely a data grain issue (possibly per-day rows not fully aggregated) rather
than 9 genuinely distinct top opportunities. Would be wrong if this is a legitimate
daily reporting artifact rather than true duplication — needs investigation before
trusting the ranking at face value.

Row 1 (content_34a70fea29d15f24): action=review_for_refresh, confidence=MEDIUM —
high impressions (39,003) but CTR is nearly zero (0.00005) despite decent position
(2.76). Genuinely looks like an opportunity: good position, high visibility, terrible
click-through. Would be wrong if this page serves a non-click intent (e.g. users get
their answer from the snippet alone).

Row 3 (content_945d6ff91386c817): action=review_for_refresh, confidence=LOW — flagged
as stale+visible, but position (8.6) is actually quite good and CTR is exactly 0 with
0 clicks despite 37k impressions. Zero clicks on 37k impressions is unusual enough to
be a data quality flag, not just a content problem. Would be wrong if this is a tracking
gap rather than a real CTR failure.

Row 8,12,14,15 (content_fec55986a1868d62, content_9c057b66c30a3abb): action=
review_for_refresh, confidence=LOW — all four have near-zero avg_position (0.08-0.18,
essentially rank #1) with zero clicks and no word_count data (<NA>). Rank #1 with zero
clicks is implausible for real search behavior — likely a data anomaly (missing
word_count supports this: incomplete content metadata). Would be wrong to act on this
without first confirming the data itself is trustworthy.

Row 17,19 (content_34a70fea29d15f24, content_1642f339bd6e7c8d): action=
review_for_refresh, confidence=MEDIUM — high impressions, near-zero CTR, but
reasonable positions (3.1, 4.1). A real case of "ranking fine, nobody's clicking" —
genuine title/snippet review candidates. Would be wrong if impressions are inflated
by irrelevant/broad queries that were never going to convert to clicks.

Row 18 (content_0e03de7680314cd5): action=review_for_refresh, confidence=MEDIUM —
25,582 impressions, decent position (2.4), CTR 0.19% — low but not zero, a more
"typical" refresh candidate than the anomalous rows above. Would be wrong if this
page's low CTR is explained by a non-commercial search intent.

Overall pattern: the rule is dominated by raw impression volume (baseline_score =
stale * visible * impressions), so it surfaces high-traffic pages regardless of
whether they're actually underperforming. Several top picks have excellent position
(near #1) and shouldn't be "refresh" candidates by any reasonable definition — the
rule conflates "high volume" with "needs review," which is its main weakness.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
print("Columns used in baseline_score:", ["gsc_impressions", "content_age_days"])
print("Confirmed: no product-flag columns in query.")

Columns used in baseline_score: ['gsc_impressions', 'content_age_days']
Confirmed: no product-flag columns in query.


Weak picks: the repeated content_hash_id (eadb33b5df496f4a, 9x in top 20) and the
four zero-click/near-rank-1 rows are the weakest picks — they look more like data
quality issues than genuine refresh candidates. The rule also over-indexes on raw
impression volume, pulling in pages with great position that don't need refreshing
at all.

Leakage check: confirmed no product flags (health_score, priority_score, action_type)
used anywhere — only gsc_impressions, gsc_clicks, gsc_avg_position, content_age_days,
word_count, ctr, all observable before any decision point. No future-window data used
— single month (2026-03) only.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.